# Módulo 1 · 04 — Redes neuronales (MLP y LSTM/GRU)

- **Modelo global** para todas las rutas, con embedding de ruta y escalado por la media de entrenamiento de cada ruta (las rutas comparten patrones de calendario; hay pocas series largas).
- **Entrada**: ventana de `ventana` días de demanda escalada + calendario de los 30 días futuros (conocido con anticipación).
- **Salida directa multihorizonte**: log-tasa de Poisson (30) y cuantil P90 (30), con P90 = media + softplus(·) ≥ media.
- **Pérdida**: NLL de Poisson (variable de conteo) + pinball τ=0.9.
- **Búsqueda**: aleatoria con presupuesto explícito (`busqueda.n_config` en el YAML); selección por pérdida de validación.

In [ ]:
# parameters
DEMO = False
N_SEMILLAS = 30
BUSCAR = True

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "modulo1_demanda"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common.seeds import fijar_semilla, semillas_experimento
from common.config import cargar_config
import m1lib as m1

fijar_semilla()
cfg = cargar_config("modulo1")
M1 = RAIZ / "modulo1_demanda"
DATA, FIG, MOD = M1 / "data", M1 / "figures", M1 / "models"
H, TAU = cfg["pronostico"]["horizonte"], cfg["pronostico"]["cuantil_alto"]
plt.rcParams["figure.figsize"] = (12, 4)

Y = pd.read_parquet(DATA / "processed" / "series_diarias.parquet")       # pasajeros/ruta/día (imputada)
cal = pd.read_parquet(DATA / "processed" / "calendario.parquet")
rutas = pd.read_csv(DATA / "processed" / "rutas.csv", dtype=str)
INI_VAL, INI_TEST = pd.Timestamp(cfg["datos"]["inicio_validacion"]), pd.Timestamp(cfg["datos"]["inicio_prueba"])
RUTAS_IDA = rutas.loc[rutas.sentido == "ida", "ruta"].tolist()
if DEMO:
    RUTAS_IDA = RUTAS_IDA[:cfg["demo"]["rutas"]]
Y_ida = Y[RUTAS_IDA]
origenes_test = m1.origenes_rodantes(Y.index, INI_TEST, H, cfg["pronostico"]["paso_origen"])
escalas_mase = {r: m1.escala_mase(Y_ida.loc[:INI_VAL - pd.Timedelta(days=1), r].dropna().values) for r in RUTAS_IDA}
print(f"{len(RUTAS_IDA)} rutas · {len(origenes_test)} orígenes de prueba: {origenes_test[0].date()} … {origenes_test[-1].date()}")

import torch, json, time
torch.set_num_threads(max(1, torch.get_num_threads()))
escala = Y_ida.loc[:INI_VAL - pd.Timedelta(days=1)].mean()
cal_cols = cal.columns
idx = Y_ida.index
orig_tr = idx[(idx < INI_VAL - pd.Timedelta(days=H))]
orig_va = idx[(idx >= INI_VAL - pd.Timedelta(days=1)) & (idx < INI_TEST - pd.Timedelta(days=H))]
if DEMO:
    cfg["modelo"]["epocas"] = cfg["demo"]["epocas"]; N_SEMILLAS = 2; BUSCAR = False

In [ ]:
def preparar(p):
    tr = m1.ventanas(Y_ida, cal, orig_tr, p["ventana"], H, escala)
    va = m1.ventanas(Y_ida, cal, orig_va, p["ventana"], H, escala)
    te = m1.ventanas(Y_ida, cal, origenes_test, p["ventana"], H, escala)
    return tr, va, te

def correr(p, seed):
    fijar_semilla(seed)
    tr, va, te = preparar(p)
    red = m1.construir_red(p["tipo"], p["ventana"], H, len(cal_cols), len(RUTAS_IDA), p["capas"], p["unidades"], p["dropout"])
    res = m1.entrenar(red, tr, va, TAU, p["lr"], cfg["modelo"]["epocas"], cfg["modelo"]["paciencia"],
                      cfg["modelo"]["batch"], cfg["modelo"]["optimizador"], p.get("weight_decay", 0.0), seed)
    return res, (tr, va, te)

## Búsqueda aleatoria de hiperparámetros

In [ ]:
rng = np.random.default_rng(42)
esp = cfg["busqueda"]["espacio"]
base_p = {k: cfg["modelo"][k] for k in ["tipo", "ventana", "capas", "unidades", "dropout", "lr", "weight_decay"]}
busqueda = []
if BUSCAR:
    for i in range(cfg["busqueda"]["n_config"]):
        p = {k: (v[rng.integers(len(v))]) for k, v in esp.items()}
        p = {k: (v.item() if hasattr(v, "item") else v) for k, v in p.items()}
        res, _ = correr(p, 42)
        busqueda.append({**p, "val_loss": min(res.hist_val), "epocas": len(res.hist_val),
                         "segundos": res.segundos, "parametros": m1.contar_parametros(res.modelo)})
        print(i, busqueda[-1])
    busqueda = pd.DataFrame(busqueda).sort_values("val_loss")
    busqueda.to_csv(DATA / "processed" / "busqueda_nn.csv", index=False)
else:
    busqueda = pd.read_csv(DATA / "processed" / "busqueda_nn.csv") if (DATA / "processed" / "busqueda_nn.csv").exists() else pd.DataFrame([{**base_p, "val_loss": np.nan}])
busqueda.head(10)

In [ ]:
if BUSCAR:
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
    for ax, k in zip(axes, ["tipo", "ventana", "unidades", "lr"]):
        busqueda.boxplot("val_loss", by=k, ax=ax); ax.set_title(k)
    fig.suptitle("Resultados de la búsqueda (pérdida de validación)"); fig.tight_layout()
    fig.savefig(FIG / "fig06_busqueda.png", dpi=150)
    print("Tiempo total de búsqueda:", round(busqueda.segundos.sum() / 60, 1), "min ·", len(busqueda), "configuraciones")

## Mejor configuración por familia (MLP y secuencial)

In [ ]:
def mejor_de(familias):
    b = busqueda[busqueda.tipo.isin(familias)]
    return {k: b.iloc[0][k] for k in base_p} if len(b) and b.val_loss.notna().any() else {**base_p, "tipo": familias[0]}
finales = {"mlp": mejor_de(["mlp"]), "secuencial": mejor_de(["lstm", "gru"])}
for f, p in finales.items():
    p["ventana"], p["capas"], p["unidades"] = int(p["ventana"]), int(p["capas"]), int(p["unidades"])
    p["lr"], p["dropout"], p["weight_decay"] = float(p["lr"]), float(p["dropout"]), float(p["weight_decay"])
json.dump(finales, open(MOD / "m1_config_final.json", "w"), indent=2)
finales

## 30 semillas por red final + curvas de aprendizaje

In [ ]:
semillas = semillas_experimento(N_SEMILLAS)
corridas, registros = [], []
for fam, p in finales.items():
    for s in semillas:
        res, (tr, va, te) = correr(p, s)
        pred, q = m1.predecir(res.modelo, te)
        xh, xf, y, sc, rr, oo = te
        for k in range(len(y)):
            registros.append(pd.DataFrame({"modelo": fam, "semilla": s, "ruta": RUTAS_IDA[rr[k]], "origen": oo[k],
                                           "fecha": pd.date_range(pd.Timestamp(oo[k]) + pd.Timedelta(days=1), periods=H),
                                           "h": np.arange(1, H + 1), "y": y[k], "pred": pred[k], "q_alto": q[k]}))
        corridas.append({"modelo": fam, "semilla": s, "mejor_epoca": res.mejor_epoca, "segundos": res.segundos,
                         "parametros": m1.contar_parametros(res.modelo)})
        if s == semillas[0]:
            torch.save(res.modelo.state_dict(), MOD / f"m1_{fam}.pt")
            plt.figure(figsize=(6, 3)); plt.plot(res.hist_train, label="entrenamiento"); plt.plot(res.hist_val, label="validación")
            plt.axvline(res.mejor_epoca, c="k", ls="--", label="parada temprana (mejor época)")
            plt.title(f"Curva de aprendizaje — {fam} ({p['tipo']})"); plt.legend(); plt.xlabel("época")
            plt.savefig(FIG / f"fig07_curva_{fam}.png", dpi=150); plt.show()
reg_nn = pd.concat(registros, ignore_index=True)
reg_nn.to_parquet(DATA / "processed" / "pred_nn.parquet")
corridas = pd.DataFrame(corridas); corridas.groupby("modelo").agg(["mean", "sum"]).round(2)

In [ ]:
# Métricas por semilla → media, desviación, mejor y peor (regla de comparación justa)
por_semilla = (reg_nn.groupby(["modelo", "semilla"])
               .apply(lambda g: pd.Series({"MASE": np.mean([m1.mase(gr.y, gr.pred, escalas_mase[r]) for r, gr in g.groupby("ruta")]),
                                           "MAE": m1.mae(g.y, g.pred), "cobertura": m1.cobertura(g.y, g.q_alto),
                                           "pinball": m1.perdida_cuantil(g.y, g.q_alto, TAU)})).reset_index())
por_semilla.to_csv(DATA / "processed" / "metricas_semillas_nn.csv", index=False)
por_semilla.groupby("modelo")[["MASE", "MAE", "cobertura", "pinball"]].agg(["mean", "std", "min", "max"]).round(3)

## Comparación con los modelos de referencia

In [ ]:
reg_base = pd.read_parquet(DATA / "processed" / "pred_baselines.parquet")
reg_base = reg_base[reg_base.ruta.isin(RUTAS_IDA)]
# Pronóstico de ensamble por familia (media de semillas) para tablas/figuras; la variabilidad va en el boxplot
reg_ens = reg_nn.groupby(["modelo", "ruta", "origen", "fecha", "h"], as_index=False)[["y", "pred", "q_alto"]].mean()
todos = pd.concat([reg_base, reg_ens], ignore_index=True)
todos.to_parquet(DATA / "processed" / "pred_todos.parquet")
tab = m1.tabla_errores(todos, escalas_mase, TAU)
display(tab.groupby("modelo")[["MAE", "RMSE", "MASE", f"cobertura_P{int(TAU*100)}", "pinball"]].mean().round(3))
m1.errores_por_bloque(todos).round(1)

In [ ]:
base_mase = tab[tab.modelo.isin(reg_base.modelo.unique())].groupby("modelo").MASE.mean()
ax = por_semilla.boxplot("MASE", by="modelo", figsize=(7, 4))
for i, (m, v) in enumerate(base_mase.items()):
    ax.axhline(v, ls="--", c=f"C{i+1}", label=m)
ax.legend(); plt.suptitle(""); ax.set_title(f"MASE en prueba — {N_SEMILLAS} semillas vs referencias")
plt.savefig(FIG / "fig08_boxplot_semillas.png", dpi=150)

In [ ]:
eh = todos.assign(ae=(todos.y - todos.pred).abs()).groupby(["modelo", "h"]).ae.mean().unstack(0)
eh.plot(marker=".", title="Figura. MAE según el horizonte (días 1–30)"); plt.xlabel("h")
plt.savefig(FIG / "fig09_error_horizonte.png", dpi=150)

In [ ]:
# Pronóstico vs real con banda P90: último origen antes de fin de año en prueba (temporada alta)
nombres = dict(zip(rutas.ruta, rutas.nombre))
o_alta = max([o for o in todos.origen.unique() if pd.Timestamp(o).month in (11, 12)], default=todos.origen.max())
fig, axes = plt.subplots(len(RUTAS_IDA), 1, figsize=(12, 2.6 * len(RUTAS_IDA)), sharex=True)
for ax, r in zip(np.atleast_1d(axes), RUTAS_IDA):
    g = todos[(todos.ruta == r) & (todos.origen == o_alta)]
    hist = Y_ida.loc[pd.Timestamp(o_alta) - pd.Timedelta(days=45):pd.Timestamp(o_alta) + pd.Timedelta(days=H), r]
    ax.plot(hist.index, hist.values, c="k", lw=1, label="real")
    for m, gm in g.groupby("modelo"):
        ax.plot(gm.fecha, gm.pred, lw=1, label=m)
    best = g[g.modelo == "secuencial"]
    ax.fill_between(best.fecha, best.pred, best.q_alto, alpha=.25, label=f"banda P{int(TAU*100)} (secuencial)")
    ax.set_title(nombres[r], fontsize=9)
np.atleast_1d(axes)[0].legend(fontsize=7, ncol=3)
fig.tight_layout(); fig.savefig(FIG / "fig10_pronostico_vs_real.png", dpi=150)

## Diagrama de arquitectura

_TODO: diagrama (capas, dimensiones, activaciones, nº parámetros). El resumen de abajo da los datos._

In [ ]:
for fam, p in finales.items():
    red = m1.construir_red(p["tipo"], p["ventana"], H, len(cal_cols), len(RUTAS_IDA), p["capas"], p["unidades"], p["dropout"])
    print(f"== {fam}: {p} · parámetros entrenables = {m1.contar_parametros(red):,}"); print(red)